In [1]:
#Extensions - potential

#ADF Test for mean reversion - for stocks and indices that trend up, use differences to model the residual
#For Big markets that do mean reversion etc
# 1. sell above MA, buy below MA
# 2. Sell if Day/Week is +ve, Buy if -ve
# 3. Combine both

#High movements - Black Swan moves
# Recent surprise, quick jumps - a glut or a squeeze which may see retracements
# Above, shorter term analysis eg Open/close gap analyses esp with inefficiencies in new datasets....
# Open and Close seem to be nearer, High/Low are wilder. Anything there?

#Arbitrage: Cross-mkts
#Analysis on High/Low differences etc between cross-markets

#New tickers esp small tickers that move wildly
#Adding new tickers at the bottom and others? Like BTC, JPYx etc

In [2]:
# !pip install hurst
# !pip install arch 
# !pip install lightgbm 
# !pip install scikit-learn 
# !pip install pmdarima
%load_ext autoreload
%autoreload 2

In [3]:
# Import the libraries
import pyautogui
import time
import pandas as pd
import numpy as np
import plotly.graph_objs as go
import os
import itertools

from datetime import date

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)  # or None for no limit

In [4]:
#Name the Forex and Metals tickers
Curr = [
    #Currencies
    "USDCAD", "USDCHF", "USDCNH", "USDCZK", "USDHUF", "USDJPY", "USDMXN", "USDNOK", "USDPLN", "USDSEK",
    "USDSGD", "USDTHB", "USDTRY", "USDZAR", "USDIDR", "USDINR", 
    'USDX.a','EURX',
    "EURUSD", "GBPUSD","NZDUSD"
    
#     #Metals
#     ,"XAGUSD.a","XAUUSD.a", "XPTUSD.a",
    
#     #ETFs
#     'AUS200.a','US30.a','US500.a','UK100.a','NAS100.a','EUSTX50.a','SPA35.a','JPN225.a', 'GER40.a','HK50.a',
#     'NETH25.a','CN50.a','SCI25.a','SWI20.a','FRA40.a', #'NOR25.a'
    
# #     # US Shares tickers
#     "AMD.US-24", "BABA.US-24", "GOOG.US-24", "AMZN.US-24", "AAPL.US-24", "BAC.US-24", "CAT.US-24",
#     "CVX.US-24", "C.US-24", "XOM.US-24", "F.US-24", "GM.US-24", "HPQ.US-24", "IBM.US-24", "INTC.US-24",
#     "JPM.US-24", "JNJ.US-24", "MCD.US-24", "META.US-24", "MSFT.US-24", "NKE.US-24", "NVDA.US-24",
#     "NFLX.US-24", "ORCL.US-24", "PFE.US-24", "PG.US-24", "SLB.US-24", "SNAP.US-24", "TSLA.US-24",
#     "BA.US-24", "KO.US-24", "DIS.US-24", "UNH.US-24", "VZ.US-24", "RTX.US-24", "V.US-24", "WMT.US-24"
    
#     #Commodities and Crypto
#     # Swap positive for short
#     ,"Wheat", 
# #     "Cocoa.a", "Cotton.a", "Sugar.a", "Corn.a",
#     #Swap negative for short
#     "BTCUSD", "SpotBrent", 
# #     "Coffee.a", "Soybeans.a", "LDSugar.a", "Cattle.a"
    
#     # AU Shares tickers - vet these before
#     "A2M.AU"
# #     "AGL.AU", "AIA.AU", "AIZ.AU", "ALD.AU", "ALQ.AU", "ALX.AU", "AMC.AU"

]

In [5]:
#To run the modules created
import sys
sys.path.append("..")                 # if needed to find project_pkg
import project_pkg.dld as dld_module  # import module object
from importlib import reload
reload(dld_module)

# call function/class inside the module
# dld_module.dld(Curr)

import project_pkg.utils as utils

In [6]:
# Analysis on the datasets
base_path = r"C:\Users\nitis\Documents\Forex\Data\New data"
res1 = {} #Define the dict

from project_pkg.utils import numpy_slope, Run, Filter, df_div, analyse, get_hurst, get_MFDFA, calculate_hurst, rolling_hurst

#Loop through all currencies, len(daily)
for name in Curr:
    path = fr"{base_path}\{name}.csv"
    df = pd.read_csv(path, sep='\t', engine='python')
    res1[name] = df

In [7]:
#Creating new dfs for sans-USD
#Gold and Silver
#One-way symbols:
#Others and crosses by inverting. Easy
symbols = ["XAGUSD.a","XAUUSD.a", 
          "EURUSD", "GBPUSD", "AUDUSD", "NZDUSD",
          "USDCAD", "USDCHF", "USDJPY", "USDMXN", "USDSGD"]
res2 = {} #Define the dict
res2 = df_div(symbols)
# print(res2)
# print(res2.keys())
# print(res2['EURCHF'])

In [8]:
#  store non-empty dfs
collected_st = []
collected_wk = []
adf_results = []

fin_res = {} #Define the dict
fin_res = res1 | res2
# print(fin_res['EURCHF'])
 # Strong set: 4 filters
    # - Close is near AT and/or 12m H/L
    # - Rolling MA crosses comparisons for 12 with various and AT
    
    # Weak set: 11 filters
    # - Close is near shorter period H/L
    # - Rolling MA crosses comparisons for 1 with various and 3/6
    # - Slopes for sudden jumps
    
#Mean-reverson tests
from statsmodels.tsa.stattools import adfuller
from hurst import compute_Hc
import matplotlib.pyplot as plt
    
for name, df in fin_res.items():
    #For strong and weak indicators
    df_st,df_wk = analyse(name, df)
    if not df_st.empty:
        collected_st.append(df_st)
    if not df_wk.empty:
        collected_wk.append(df_wk)
    # Tests - 
    # ADF to check Stationary or not + 
    # Hurst (x3), and MFDFA to test if mean-reversion (values less than 0.5) or trending (higher values)
    # Price close data selection
    series = df['<CLOSE>'].dropna()
    series = series[np.isfinite(series)] 
    # Tests to run
    result = adfuller(series, autolag='AIC') #ADF
    calc_h, calc_rsq_h = calculate_hurst(series.values) #From Fractcal site
    #print(name)
    avg_h, std_h = utils.rolling_hurst(series.values) #Rolling hurst summary stats
    
    # Add the data into one place to join later
    #Hurst:<0.5 means high->low; > means high->high. 
    #MFDFA: Monofractal is more trustworthy, multifractal width may have some fakeouts too
    adf_results.append({
        'Currency': name,
        'ADF_pvalue': result[1], #P-value to reject the null of no mean rev. for ADF
        'ADF_Statistic': result[0],
        'Hurst': get_hurst(series), 
        'MFDFA': get_MFDFA(series.values), 
        'calc_h': calc_h,
        'avg_h': avg_h,
        'std_h': std_h
    })

# Convert the whole list to a DF in one shot
adf_results = pd.DataFrame(adf_results)

#Still empty checks
if collected_st:
    result_st = pd.concat(collected_st, ignore_index=True)
else:
    result_st = pd.DataFrame()  # empty result
if collected_wk:
    result_wk = pd.concat(collected_wk, ignore_index=True)
else:
    result_wk = pd.DataFrame()  # empty result

#Final df
df_fin_0 = result_st.merge(result_wk, on='Currency', how='outer')
df_fin = df_fin_0.merge(adf_results, on='Currency', how='outer')
df_fin.rename(columns={'Total_Filters_x': 'St_Filters','Total_Filters_y': 'Wk_Filters'
                      ,'AT_Low_y': 'AT_Low','AT_High_y': 'AT_High','<CLOSE>_y': '<CLOSE>'
                      ,'Close_check_y': 'Close_check'}
                      , inplace=True)
df_fin = df_fin.sort_values(by=['St_Filters','Wk_Filters'], ascending=False).reset_index(drop=True) 

C:\Users\nitis\Documents\GitHub\sukhiahlu\Trading Py\Neo trades\project_pkg\utils.py:233: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df2['Close_check'] = close_check(df2)
C:\Users\nitis\Documents\GitHub\sukhiahlu\Trading Py\Neo trades\project_pkg\utils.py:233: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df2['Close_check'] = close_check(df2)
C:\Users\nitis\Documents\GitHub\sukhiahlu\Trading Py\Neo trades\project_pkg\utils.py:233: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice 

C:\Users\nitis\Documents\GitHub\sukhiahlu\Trading Py\Neo trades\project_pkg\utils.py:233: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df2['Close_check'] = close_check(df2)
C:\Users\nitis\Documents\GitHub\sukhiahlu\Trading Py\Neo trades\project_pkg\utils.py:233: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df2['Close_check'] = close_check(df2)
C:\Users\nitis\Documents\GitHub\sukhiahlu\Trading Py\Neo trades\project_pkg\utils.py:233: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice 

C:\Users\nitis\Documents\GitHub\sukhiahlu\Trading Py\Neo trades\project_pkg\utils.py:233: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df2['Close_check'] = close_check(df2)
C:\Users\nitis\Documents\GitHub\sukhiahlu\Trading Py\Neo trades\project_pkg\utils.py:233: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df2['Close_check'] = close_check(df2)
C:\Users\nitis\Documents\GitHub\sukhiahlu\Trading Py\Neo trades\project_pkg\utils.py:233: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice 

C:\Users\nitis\Documents\GitHub\sukhiahlu\Trading Py\Neo trades\project_pkg\utils.py:233: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df2['Close_check'] = close_check(df2)
C:\Users\nitis\Documents\GitHub\sukhiahlu\Trading Py\Neo trades\project_pkg\utils.py:233: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df2['Close_check'] = close_check(df2)
C:\Users\nitis\Documents\GitHub\sukhiahlu\Trading Py\Neo trades\project_pkg\utils.py:233: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice 

In [1]:
%history -g -f recovered_code_v2.py

In [9]:
#Option to analyse manually
path = fr"C:\Users\nitis\Desktop\manual.txt"
# df_fin = pd.read_fwf(path) #For txt file load

#Drop redundant columns and #Order columns in the correct spot
df_fin = df_fin[['Currency','St_Filters','Wk_Filters','AT_High','AT_Low','<CLOSE>',
                 'Close_check','ADF_pvalue','Hurst','MFDFA','calc_h','avg_h','std_h']]

df_fin['diff_hu'] = df_fin['avg_h'] - df_fin['calc_h']

#Get the ADF best + St_Filters
print(df_fin[(df_fin['avg_h']<0.45)]) #& (df_fin['ADF_pvalue']<0.11)]) #& (df_fin['Wk_Filters']>1) 
print(len(fin_res))

from datetime import date

# Get today's date
today = date.today()

# Format the date as 'DD Mon' (e.g., '02 Nov')
date_string = today.strftime('%d %b')

filename = f"List_{date_string}.txt"

# Create the text output string
text_output = df_fin.to_string(index=False, header=True)

# 2. DEFINE THE FULL PATH
import os
base_path = r'C:\Users\nitis\Desktop'
base_path2 = r'C:\Users\nitis\Desktop\Fin stuff\Week output'

full_path = os.path.join(base_path, filename)
full_path2 = os.path.join(base_path2, filename)

# Write the string directly to the file
with open(full_path, 'w', encoding='utf-8') as f:
    f.write(text_output)

# Write the string directly to the file
with open(full_path2, 'w', encoding='utf-8') as f:
    f.write(text_output)

Empty DataFrame
Columns: [Currency, St_Filters, Wk_Filters, AT_High, AT_Low, <CLOSE>, Close_check, ADF_pvalue, Hurst, MFDFA, calc_h, avg_h, std_h, diff_hu]
Index: []
76


In [7]:
%load_ext autoreload
%autoreload 2

# To run the modules created
from project_pkg.model_v3 import run_walk_forward_validation, predict_next_1_day
import pandas as pd
import warnings

# IMPORT VALUEWARNING SO PYTHON RECOGNIZES IT
from statsmodels.tools.sm_exceptions import ValueWarning

# 1. Force Python to silence the specific statsmodels base warnings
warnings.filterwarnings("ignore", category=ValueWarning, module="statsmodels")
warnings.filterwarnings("ignore", category=FutureWarning, module="statsmodels")

# Quick catch-all backup if the module path names differ slightly in your environment:
warnings.filterwarnings("ignore", message=".*No supported index is available.*")

# 2. Load the data
df_mod = pd.read_csv(r"C:\Users\nitis\Documents\Forex\Data\New data\audusd.csv", sep='\t', engine='python')

# Unpack the updated variable outputs in your cell
processed_df, historical_accuracy, ranked_features = run_walk_forward_validation(df_mod, train_window=250, test_steps=20)

print(f"Historical Backtest Accuracy: {historical_accuracy:.2%}")

print("\n--- ALL RANKED FEATURES ---")
for rank, (feature, importance) in enumerate(ranked_features, 1):
    print(f"Rank {rank:02d}: {feature:<15} -> Impact: {importance:.2f}%")

predict_next_1_day(processed_df, train_window=250)

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
Rows remaining after technical indicator cleanup: 1100
Pre-computing ARIMA/GARCH features (Optimized: updating every 5 days)...
Rows remaining after ARIMA/GARCH statistical cleanup: 849
Starting walk-forward validation for 20 iterations...
Historical Backtest Accuracy: 60.00%

--- ALL RANKED FEATURES ---
Rank 01: RSI             -> Impact: 31.43%
Rank 02: ROC_10          -> Impact: 22.23%
Rank 03: NTR_5           -> Impact: 13.84%
Rank 04: dist_MA_20      -> Impact: 12.97%
Rank 05: volume_ratio    -> Impact: 8.12%
Rank 06: Stochastic_14   -> Impact: 6.20%
Rank 07: dist_MA_5       -> Impact: 4.22%
Rank 08: HL_spread       -> Impact: 0.58%
Rank 09: garch_pred      -> Impact: 0.38%
Rank 10: arima_pred      -> Impact: 0.04%
Rank 11: volatility_5    -> Impact: 0.00%
Rank 12: CO_spread       -> Impact: 0.00%

      1-DAY FORWARD DIRECTIONAL FORECAST          
Current Date (T):   2026-05-22
Forecast Date (

In [6]:
%load_ext autoreload
%autoreload 2

# To run the modules created
from project_pkg.model import (
    calculate_technical_indicators,
    run_walk_forward_validation_tuned, 
    predict_next_1_day
)

import pandas as pd
import warnings

# IMPORT VALUEWARNING SO PYTHON RECOGNIZES IT
from statsmodels.tools.sm_exceptions import ValueWarning

# 1. Force Python to silence the specific statsmodels base warnings
warnings.filterwarnings("ignore", category=ValueWarning, module="statsmodels")
warnings.filterwarnings("ignore", category=FutureWarning, module="statsmodels")

# Quick catch-all backup if the module path names differ slightly in your environment:
warnings.filterwarnings("ignore", message=".*No supported index is available.*")

# 2. Load the data
df_mod = pd.read_csv(r"C:\Users\nitis\Documents\Forex\Data\New data\audusd.csv", sep='\t', engine='python')


# --- RUN THE TUNED BACKTEST ---
# train_window=250 means the model looks at roughly 1 year of daily history to learn patterns
# test_steps=50 means it will simulate trading for the last 50 consecutive market sessions
processed_df, historical_accuracy, ranked_features = run_walk_forward_validation_tuned(
    df=df_mod, 
    train_window=250, 
    test_steps=50
)

print("\n" + "="*50)
print("          TUNED BACKTEST RESULTS                  ")
print("="*50)
print(f"Adaptive Directional Accuracy: {historical_accuracy:.2%}")
print("="*50)
print("\n          COMPLETE FEATURE IMPORTANCE RANKINGS     ")
print("="*50)
for rank, (feature, importance) in enumerate(ranked_features, 1):
    print(f"Rank {rank:02d}: {feature:<15} -> Relative Impact: {importance:.2f}%")
print("="*50)

# --- GENERATE THE FORECAST FOR TOMORROW ---
# This uses the absolute latest data point to give you a live prediction
predict_next_1_day(processed_df, train_window=250)

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
Executing robust baseline engine backtest across 50 synchronized sessions...

          TUNED BACKTEST RESULTS                  
Adaptive Directional Accuracy: 58.00%

          COMPLETE FEATURE IMPORTANCE RANKINGS     
Rank 01: RSI             -> Relative Impact: 36.31%
Rank 02: ROC_10          -> Relative Impact: 18.41%
Rank 03: HL_spread       -> Relative Impact: 11.87%
Rank 04: NTR_5           -> Relative Impact: 8.61%
Rank 05: Stochastic_14   -> Relative Impact: 8.01%
Rank 06: dist_MA_20      -> Relative Impact: 5.34%
Rank 07: dist_MA_5       -> Relative Impact: 3.88%
Rank 08: volatility_5    -> Relative Impact: 3.58%
Rank 09: volume_ratio    -> Relative Impact: 2.92%
Rank 10: CO_spread       -> Relative Impact: 1.07%

      1-DAY FORWARD DIRECTIONAL FORECAST          
Current Date (T):   2026-05-22
Forecast Date (T+1): 2026-05-25
Up Probability:      56.7%
Nudge Action:        ⚪ COIN FLIP / NO

In [2]:
# Appendix below on other currencies/stocks to try

In [12]:
# # Other shares, softs, hards, etc

# # AU Shares tickers
# au_share_tickers = [
#     "A2M.AU", "ABC.AU", "AGL.AU", "AIA.AU", "AIZ.AU", "ALD.AU", "ALQ.AU", "ALU.AU", "ALX.AU", "AMC.AU",
#     "AMP.AU", "ANN.AU", "ANZ.AU", "APA.AU", "APE.AU", "APX.AU", "ARG.AU", "ARB.AU", "ASB.AU", "ASX.AU",
#     "AST.AU", "AUB.AU", "AWB.AU", "AWL.AU", "AWN.AU", "AWC.AU", "BEN.AU", "BGA.AU", "BHP.AU", "BIN.AU",
#     "BKL.AU", "BKW.AU", "BLD.AU", "BOQ.AU", "BPT.AU", "BRG.AU", "BSL.AU", "BVS.AU", "BWP.AU", "BXB.AU",
#     "CAR.AU", "CBA.AU", "CCP.AU", "CCL.AU", "CGC.AU", "CGF.AU", "CHC.AU", "CIP.AU", "CIM.AU", "CIA.AU",
#     "CLW.AU", "CMW.AU", "CNU.AU", "COL.AU", "COE.AU", "COH.AU", "CPU.AU", "CQR.AU", "CRN.AU", "CSL.AU",
#     "CSR.AU", "CTD.AU", "CUV.AU", "CWN.AU", "CWY.AU", "DHG.AU", "DMP.AU", "DOW.AU", "DDR.AU", "DEG.AU",
#     "DRR.AU", "DXS.AU", "EBO.AU", "EHE.AU", "EHL.AU", "ELD.AU", "EHE.AU", "EML.AU", "EVT.AU", "EVN.AU",
#     "FBU.AU", "FLT.AU", "FMG.AU", "FPH.AU", "GEM.AU", "GNC.AU", "GMG.AU", "GOR.AU", "GOZ.AU", "GPT.AU",
#     "GQG.AU", "GUD.AU", "GWA.AU", "HDN.AU", "HLS.AU", "HMC.AU", "HUB.AU", "HVN.AU", "IAG.AU", "IFT.AU",
#     "IEL.AU", "IFM.AU", "IFL.AU", "ILU.AU", "INA.AU", "INC.AU", "IRE.AU", "IPL.AU", "IPH.AU", "IVC.AU",
#     "JBH.AU", "JHX.AU", "JHG.AU", "JIN.AU", "KGN.AU", "LNK.AU", "LIS.AU", "LLC.AU", "LOV.AU", "LTR.AU",
#     "LYC.AU", "MFG.AU", "MIN.AU", "MGR.AU", "MMS.AU", "MPL.AU", "MP1.AU", "MQG.AU", "MTS.AU", "MYX.AU",
#     "NAN.AU", "NAB.AU", "NEC.AU", "NEU.AU", "NHC.AU", "NHF.AU", "NIC.AU", "NEM.AU", "NCM.AU", "NEA.AU",
#     "NUF.AU", "NSR.AU", "NWH.AU", "NXT.AU", "NWL.AU", "ORA.AU", "ORG.AU", "ORI.AU", "OML.AU", "ORE.AU",
#     "OZL.AU", "PDL.AU", "PDN.AU", "PER.AU", "PLS.AU", "PME.AU", "PMV.AU", "PNI.AU", "PNV.AU", "PPK.AU",
#     "PPT.AU", "PRN.AU", "PRU.AU", "PTM.AU", "PXA.AU", "QAN.AU", "QBE.AU", "QUB.AU", "RBL.AU", "REA.AU",
#     "REH.AU", "REG.AU", "RHC.AU", "RIO.AU", "RMD.AU", "RRL.AU", "RSG.AU", "RWC.AU", "S32.AU", "SAR.AU",
#     "SBM.AU", "SCA.AU", "SCG.AU", "SCP.AU", "SDG.AU", "SDF.AU", "SEK.AU", "SFR.AU", "SGM.AU", "SHL.AU",
#     "SIQ.AU", "SKC.AU", "SKI.AU", "SLC.AU", "SLR.AU", "SML.AU", "SMR.AU", "SOL.AU", "SPK.AU", "SSG.AU",
#     "STO.AU", "SUL.AU", "SUN.AU", "SVW.AU", "SWM.AU", "SYD.AU", "SXL.AU", "TAH.AU", "TCL.AU", "TGR.AU",
#     "TLS.AU", "TLX.AU", "TNE.AU", "TPG.AU", "TWE.AU", "URW.AU", "VCX.AU", "VEA.AU", "VOC.AU", "VNT.AU",
#     "VUK.AU", "WBC.AU", "WEB.AU", "WES.AU", "WMC.AU", "WOR.AU", "WOW.AU", "WPL.AU", "WPR.AU", "WSA.AU",
#     "WTC.AU", "XRO.AU", "YAL.AU", "Z1P.AU"
# ]

# # Combine all tickers and sort
# all_tickers = sorted(au_share_tickers + us_share_tickers + forex_metals_tickers)

# # Create DataFrame and export to CSV
# df = pd.DataFrame(all_tickers, columns

In [13]:
#### For charts on the Hurst
# import matplotlib.pyplot as plt
# from hurst import compute_Hc

# # H = the slope, c = the intercept, data = (list of window sizes, list of R/S values)
# H, c, data = compute_Hc(series, kind='price')

# # The 'data' variable contains exactly what you need to plot
# window_sizes = data[0]
# RS_values = data[1]

# plt.loglog(window_sizes, RS_values, 'o', label='Data Points')
# plt.loglog(window_sizes, c * window_sizes**H, label=f'Hurst Line (H={H:.2f})')
# plt.title("Hurst Interrogation Plot")
# plt.xlabel("Log(Window Size)")
# plt.ylabel("Log(R/S)")
# plt.legend()
# plt.show()